# Read the incidents in Mongo

Pipeline step 3 — a live read of what the coding **app** is writing to Atlas.
Run the **connect** cell once, then re-run **Overview** / **Detail** any time to
refresh as you code in the app.

One document per incident (keyed by `incident_id`); each source document's coding
lives under `by_document.<doc_key>` with its `fields`, `quotes`, and `claims`.

In [ ]:
import os
from pathlib import Path
from pymongo import MongoClient

# Load .env (git-ignored) without any dependency, so the password stays out of
# this notebook and out of version control.
env = Path(".env")
if env.exists():
    for line in env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

uri = os.environ["MONGO_URI"]                       # from .env or your shell env
DB_NAME = os.environ.get("MONGO_DB", "incidents")
client = MongoClient(uri, serverSelectionTimeoutMS=5000)
client.admin.command("ping")
db = client[DB_NAME]
incidents = db["incidents"]
print(f"✓ Connected to {DB_NAME!r} — {incidents.count_documents({})} incident(s) in the collection")

In [ ]:
# Overview — re-run this cell any time to see the current state as the app writes.
import pandas as pd

rows = []
for d in incidents.find():
    bydoc = d.get("by_document", {})
    rows.append({
        "incident_id": d.get("incident_id"),
        "title": (d.get("incident_title") or "")[:60],
        "docs": len(d.get("documents", [])),
        "quotes": sum(len(c.get("quotes", [])) for c in bydoc.values()),
        "claims": sum(len(c.get("claims", [])) for c in bydoc.values()),
        "updated_at": d.get("updated_at"),
    })

df = (pd.DataFrame(rows).sort_values("updated_at", ascending=False, ignore_index=True)
      if rows else pd.DataFrame(columns=["incident_id", "title", "docs", "quotes", "claims", "updated_at"]))
print(f"{len(df)} incident(s) in Mongo")
df

In [ ]:
# Detail — drill into one incident's coding.
from pprint import pprint

inc = incidents.find_one()                              # or: {"incident_id": "INC-001"}
if not inc:
    print("No incidents yet — code a document in the app and re-run.")
else:
    print("incident_id:", inc["incident_id"], "|", inc.get("incident_title"))
    print("documents:", [doc["doc_id"] for doc in inc.get("documents", [])])
    for doc_key, coding in inc.get("by_document", {}).items():
        answers = {k: v.get("answer") for k, v in coding.get("fields", {}).items() if v.get("answer")}
        print(f"\n— {doc_key} —  ({len(coding.get('quotes', []))} quotes, {len(coding.get('claims', []))} claims)")
        pprint(answers)